In [ ]:
# Cell 1 — Objective: Install only the required packages for this notebook; import nothing sensitive here.
# Notes:
# - This cell prepares the local Jupyter kernel only. No OCI or cloud calls are made here.
# - Keep the dependency set minimal for this generators-only workflow.

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets ipyfilechooser paramiko locust

print("Cell 1 complete: Python environment ready (Generators-only; control-plane SSH pinned to IPv4).")
print("NEXT: Run Cell 2 to load central variables (no network calls) and set OUTPUT_DIR/TS_UTC.")


In [ ]:
# Cell 2 — Objective: Central variables and credentials (single source of truth; no network calls)
# - Centralize all configurable values; validate required files; do not print secrets.
# - Create OUTPUT_DIR and TS_UTC for consistent artifact naming.

import os
from datetime import datetime, timezone
import oci

# OCI config (no network calls)
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE     = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)  # local validation only

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg["tenancy"])
USER_OCID    = os.environ.get("USER_OCID",    _cfg["user"])
FINGERPRINT  = os.environ.get("FINGERPRINT",  _cfg["fingerprint"])

OCI_PRIVATE_KEY_PATH   = os.path.expanduser(os.environ.get("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", "")))
PRIVATE_KEY_PASSPHRASE = os.environ.get("OCI_PASSPHRASE", os.environ.get("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")))
REGION                 = os.environ.get("REGION", _cfg["region"])

def _ensure_path_exists(path: str, label: str):
    if not path:
        raise FileNotFoundError(f"{label} path not set.")
    p = os.path.expanduser(path)
    if not os.path.exists(p):
        raise FileNotFoundError(f"{label} not found: {p}")

# Fail-fast on required private key presence (do not log secret content)
_ensure_path_exists(OCI_PRIVATE_KEY_PATH, "OCI private key")

# SSH keys (defaults; can be changed later in UI)
_default_ssh_pub  = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub"))
_default_ssh_priv = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa"))
SSH_PUBLIC_KEY_PATH  = _default_ssh_pub  if os.path.exists(_default_ssh_pub)  else ""
SSH_PRIVATE_KEY_PATH = _default_ssh_priv if os.path.exists(_default_ssh_priv) else ""

# Generators configuration (defaults; overridable via env and UI)
GENERATOR_COUNT     = int(os.environ.get("GENERATOR_COUNT", "4"))
GENERATOR_SHAPE     = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS     = float(os.environ.get("GENERATOR_OCPUS", "8"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "32"))

# IPv6 toggles (IPv4 default; SSH pinned to IPv4 later)
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")
USE_SSH_IPV6              = False  # pinned to IPv4 for control-plane

# SSH ingress CIDRs (both families; v6 unused for control-plane in this plan)
SSH_ALLOWED_CIDR    = os.environ.get("SSH_ALLOWED_CIDR", "0.0.0.0/0")
SSH_ALLOWED_V6_CIDR = os.environ.get("SSH_ALLOWED_V6_CIDR", "::/0").strip()

# Locust and UI settings
LOCUST_WORKDIR           = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")
TEST_MODE                = os.environ.get("TEST_MODE", "cps").strip().lower()   # "cps" | "throughput"
LOCUST_WAIT_TIME_SEC     = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
LOCUST_CONNECT_TIMEOUT   = int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))   # ms
LOCUST_READ_TIMEOUT      = int(os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000"))    # ms
LOCUST_VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
HEALTH_ENDPOINT_PATH     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/")

UI_WEB_PORT = int(os.environ.get("UI_WEB_PORT", "8089"))

# CPU → workers policy
WORKERS_PER_HOST     = os.environ.get("WORKERS_PER_HOST", "auto")  # "auto" or "fixed" (numeric set later)
CPU_RESERVE          = int(os.environ.get("CPU_RESERVE", "1"))
MIN_WORKERS_PER_HOST = int(os.environ.get("MIN_WORKERS_PER_HOST", "1"))
MAX_WORKERS_PER_HOST = int(os.environ.get("MAX_WORKERS_PER_HOST", "32"))

# External targets (comma-separated URLs; normalized later)
EXTERNAL_TARGETS_TEXT = os.environ.get("EXTERNAL_TARGETS_TEXT", "").strip()

# Output directory & UTC timestamp
OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
TS_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Placeholders to be set by UI (Cell 3) and used by later cells
SELECTED_REGION = None
COMPARTMENT_ID  = ""
AD_A            = ""
IMAGE_ID        = ""

print(f"Cell 2 complete: Config loaded from {OCI_CONFIG_FILE} [{OCI_PROFILE}] — region={REGION}")
print("Defaults applied. IPv4 control-plane; IPv6 assignment optional in Cell 3.")
print("NEXT: Run Cell 3 to choose region/compartment/AD, generator shape/count, SSH keys, IPv6 assignment, and Locust/UI basics; then Apply.")


In [ ]:
# Cell 3 — Objective: Simple, visible UI to choose region/compartment/AD, generator count/shape (Flex shows OCPUs/Mem),
# SSH keys, IPv6 assignment, Locust/UI basics, workers policy, and external targets; Apply persists selections.
# Visual polish:
# - Labels never truncate (description_width='initial' + CSS wrapping).
# - Consistent row widths and spacing for readability.
# - Non-blocking validation with a compact issues strip.
# - OCPUs/Memory fields visible ONLY when shape endswith(".Flex").
# - No secrets printed.

import os
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as widgets
import oci

SECTION_BG = "#f8f9fb"
BORDER = "1px solid #e0e0e0"
PAD = "12px"
CONTAINER_MAX_W = "96%"

# Light CSS to improve label wrapping and issues strip styling
_css = HTML("""
<style>
  /* Ensure widget labels wrap instead of truncating */
  .widget-label { white-space: normal !important; line-height: 1.2; }
  /* Compact inline hints */
  .hint { color: #555; font-size: 12px; margin: 2px 0 0 0; }
  /* Issues strip (non-blocking validation summary) */
  .issues-strip {
    background: #fdecea; color: #ba1a1a;
    padding: 6px 10px; border: 1px solid #f5c2c7; border-radius: 6px;
    font-size: 13px; margin: 0 0 6px 0;
  }
  /* Loading labels */
  .loading-note { color: #555; font-size: 12px; }
  /* Section card */
  .nb-section { border-left: 4px solid #d2d7e5; }
</style>
""")

def section(title_text, body_widgets):
    title = widgets.HTML(f"<b>{title_text}</b>")
    body = body_widgets if isinstance(body_widgets, widgets.Widget) else widgets.VBox(body_widgets)
    box = widgets.VBox([title, body], layout=widgets.Layout(width="100%", border=BORDER, padding=PAD, margin="8px 0", background_color=SECTION_BG))
    try:
        box.add_class("nb-section")
    except Exception:
        pass
    return box

def row(*children, gap="12px", widths=None):
    # widths: optional list of CSS width strings matching children
    if widths and len(widths) == len(children):
        for w, cw in zip(children, widths):
            try:
                w.layout.width = cw
            except Exception:
                pass
    hb = widgets.HBox(list(children), layout=widgets.Layout(gap=gap, width="100%", align_items="flex-start"))
    return hb

def build_signer(tenancy, user, fp, key_path, passphrase, cfg):
    return oci.signer.Signer(
        tenancy=tenancy,
        user=user,
        fingerprint=fp,
        private_key_file_location=key_path,
        pass_phrase=passphrase if passphrase else None,
        private_key_content=cfg.get("key_content"),
    )

def cfg_for_region(region_name):
    c = dict(_cfg)
    c["region"] = region_name
    return c

# Region dropdown (from subscriptions)
_base_signer = build_signer(TENANCY_OCID, USER_OCID, FINGERPRINT, OCI_PRIVATE_KEY_PATH, PRIVATE_KEY_PASSPHRASE, _cfg)
idc_base = oci.identity.IdentityClient(config=_cfg, signer=_base_signer)
subs = sorted(idc_base.list_region_subscriptions(TENANCY_OCID).data, key=lambda r: r.region_name)
region_options = [(r.region_name, r.region_name) for r in subs]
default_region = (REGION if REGION in [r.region_name for r in subs] else (next((r.region_name for r in subs if r.is_home_region), subs[0].region_name)))
region_dd = widgets.Dropdown(options=region_options, value=default_region, description="Region:", layout=widgets.Layout(width="100%"))
reload_btn = widgets.Button(description="Reload", icon="refresh")
reset_btn  = widgets.Button(description="Reset",  icon="history")

# Outputs and dynamic container
err_out, summary_out = widgets.Output(), widgets.Output()
dynamic_box = widgets.VBox([])

# Small helper to discover files
def _discover_files(dirs, exts=None, include_hidden=True):
    out = []
    for d in dirs:
        p = Path(os.path.expanduser(d))
        if not p.exists() or not p.is_dir():
            continue
        for f in p.iterdir():
            if not f.is_file():
                continue
            if not include_hidden and f.name.startswith("."):
                continue
            if exts is not None and not any(str(f).endswith(ext) for ext in exts):
                continue
            out.append(str(f))
    out.sort(key=lambda s: Path(s).stat().st_mtime if Path(s).exists() else 0, reverse=True)
    return out

# SSH keys dropdowns
ssh_dir = os.path.expanduser("~/.ssh")
ssh_pub_candidates  = [p for p in _discover_files([ssh_dir], exts=[".pub"])]
ssh_priv_candidates = [p for p in _discover_files([ssh_dir], exts=None) if not p.endswith(".pub")]
ssh_pub_default  = next((p for p in ssh_pub_candidates if p == SSH_PUBLIC_KEY_PATH),  (ssh_pub_candidates[0] if ssh_pub_candidates else ""))
ssh_priv_default = next((p for p in ssh_priv_candidates if p == SSH_PRIVATE_KEY_PATH), (ssh_priv_candidates[0] if ssh_priv_candidates else ""))

ssh_pub_dd  = widgets.Dropdown(options=[(p, p) for p in ssh_pub_candidates]  or [("No *.pub keys in ~/.ssh", "")], value=ssh_pub_default,  description="SSH pub:",  layout=widgets.Layout(width="100%"))
ssh_priv_dd = widgets.Dropdown(options=[(p, p) for p in ssh_priv_candidates] or [("No private keys in ~/.ssh", "")], value=ssh_priv_default, description="SSH priv:", layout=widgets.Layout(width="100%"))

# Generators core inputs
gen_count_in  = widgets.BoundedIntText(value=int(GENERATOR_COUNT), min=1, max=256, step=1, description="Generators:")
shape_filter  = widgets.Text(value="", description="Shape filter:", placeholder="e.g. E5.Flex", layout=widgets.Layout(width="100%"))

# Validation labels (inline under specific controls)
lbl_err_shape     = widgets.HTML("")
lbl_err_image     = widgets.HTML("")
lbl_err_gen_count = widgets.HTML("")
lbl_err_ssh_pub   = widgets.HTML("")
lbl_err_ssh_priv  = widgets.HTML("")
lbl_err_flex      = widgets.HTML("")
lbl_err_workers   = widgets.HTML("")
issues_strip      = widgets.HTML("")  # compact issues list above Apply
try:
    issues_strip.add_class("issues-strip")
except Exception:
    # Fallback styling without add_class
    issues_strip.value = "<div style='background:#fdecea;color:#ba1a1a;padding:6px 10px;border:1px solid #f5c2c7;border-radius:6px;font-size:13px;display:none;'></div>"

# Helper to set/clear error text
def _err(lbl: widgets.HTML, msg: str | None):
    lbl.value = f"<span style='color:#b00020'>{msg}</span>" if msg else ""

# State bag for widgets used in Apply
_state = {"comp_dd": None, "ad_dd": None, "gen_shape_dd": None, "gen_ocpus_in": None, "gen_mem_in": None, "image_dd": None}

# Clients per region
def identity_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(TENANCY_OCID, USER_OCID, FINGERPRINT, OCI_PRIVATE_KEY_PATH, PRIVATE_KEY_PASSPHRASE, cfg_r)
    return oci.identity.IdentityClient(config=cfg_r, signer=signer)

def compute_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(TENANCY_OCID, USER_OCID, FINGERPRINT, OCI_PRIVATE_KEY_PATH, PRIVATE_KEY_PASSPHRASE, cfg_r)
    return oci.core.ComputeClient(config=cfg_r, signer=signer)

def list_compartments(idc):
    comps = oci.pagination.list_call_get_all_results(idc.list_compartments, TENANCY_OCID, compartment_id_in_subtree=True, access_level="ACCESSIBLE").data
    comps = [c for c in comps if c.lifecycle_state == "ACTIVE"]
    tenancy = idc.get_tenancy(TENANCY_OCID).data
    tenancy_name = getattr(tenancy, "name", "root-tenancy")
    return [(f"{tenancy_name} (root)", TENANCY_OCID)] + sorted([(c.name + f" ({c.description or 'no-desc'})", c.id) for c in comps], key=lambda t: t[0].lower())

def list_ads(idc):
    ads = oci.pagination.list_call_get_all_results(idc.list_availability_domains, TENANCY_OCID).data
    return sorted([ad.name for ad in ads]) or ["AD-1"]

def list_shapes(cc):
    shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
    return sorted({s.shape for s in shapes})

# Ensure Flex widgets exist
def _ensure_flex_inputs():
    if _state["gen_ocpus_in"] is None:
        _state["gen_ocpus_in"] = widgets.BoundedIntText(value=int(GENERATOR_OCPUS),     min=1, max=256, step=1, description="OCPUs (Flex):")
        _state["gen_mem_in"]   = widgets.BoundedIntText(value=int(GENERATOR_MEMORY_GB), min=1, max=4096, step=1, description="Memory GB (Flex):")

# Show/hide Flex inputs based on shape selection (only visible when shape endswith ".Flex")
def _toggle_flex(shape_name: str):
    _ensure_flex_inputs()
    is_flex = (shape_name or "").endswith(".Flex")
    disp = "" if is_flex else "none"
    _state["gen_ocpus_in"].layout.display = disp
    _state["gen_mem_in"].layout.display   = disp
    if not is_flex:
        _err(lbl_err_flex, "")  # clear flex error when hiding

# IPv6 assignment (SSH pinned to IPv4)
enable_v6_cb = widgets.Checkbox(value=ENABLE_IPV6_ON_GENERATORS, description="Assign IPv6 to generators")
ssh_ipv6_pinned_lbl = widgets.HTML("<i>SSH to generators is pinned to IPv4 for reliability.</i>")

ssh_v4_cidr_in = widgets.Text(value=SSH_ALLOWED_CIDR,    description="SSH v4 CIDR:")
ssh_v6_cidr_in = widgets.Text(value=SSH_ALLOWED_V6_CIDR, description="SSH v6 CIDR:")

# Locust & UI settings
test_mode_dd  = widgets.Dropdown(options=[("CPS (connections/sec)", "cps"), ("Throughput (GET payload)", "throughput")], value=TEST_MODE, description="Test mode:")
wait_time_in  = widgets.FloatText(value=LOCUST_WAIT_TIME_SEC, description="Wait(s)/user:", step=0.1)
conn_to_in    = widgets.BoundedIntText(value=LOCUST_CONNECT_TIMEOUT, min=1000, max=60000,  step=500, description="Connect ms:")
read_to_in    = widgets.BoundedIntText(value=LOCUST_READ_TIMEOUT,    min=1000, max=120000, step=500, description="Read ms:")
verify_tls_in = widgets.Checkbox(value=LOCUST_VERIFY_TLS, description="Verify TLS")
ui_port_in    = widgets.BoundedIntText(value=UI_WEB_PORT, min=1024, max=65535, step=1, description="UI port:")

# External targets + live normalized preview
targets_in = widgets.Textarea(
    value=EXTERNAL_TARGETS_TEXT,
    description="External targets:",
    layout=widgets.Layout(width="100%", height="80px"),
    placeholder="Comma-separated URLs or hosts/IPs, e.g. https://vip1, https://[2001:db8::1]"
)
targets_preview = widgets.HTML("<small class='hint'>Normalized: (none)</small>")
targets_hint = widgets.HTML("<small class='hint'>Scheme required (https:// or http://); bracket IPv6 literals, e.g., https://[2001:db8::1]</small>")

def _normalize_targets(txt: str) -> list[str]:
    out = []
    for tok in (txt or "").split(","):
        t = (tok or "").strip()
        if not t:
            continue
        if not (t.startswith("http://") or t.startswith("https://")):
            t = "https://" + t
        try:
            scheme, rest = t.split("://", 1)
            host = rest.split("/")[0]
            if ":" in host and not (host.startswith("[") and host.endswith("]")):
                t = f"{scheme}://[{host}]" + rest[len(host):]
        except Exception:
            pass
        if t not in out:
            out.append(t)
    return out

def _update_targets_preview(*_):
    norm = _normalize_targets(targets_in.value or "")
    targets_preview.value = "<small class='hint'>Normalized: " + (", ".join(norm) if norm else "(none)") + "</small>"

targets_in.observe(_update_targets_preview, names="value")
_update_targets_preview()

# Workers per host
def _default_mode_from_value(v):
    if isinstance(v, str) and v.strip().lower() == "auto":
        return "auto"
    try:
        if isinstance(v, int):
            return "fixed"
        if isinstance(v, str) and v.strip().isdigit():
            return "fixed"
    except Exception:
        pass
    return "auto"

workers_per_host_mode_in = widgets.Dropdown(
    options=[("Auto (use OCPUs)", "auto"), ("Fixed count", "fixed")],
    value=_default_mode_from_value(WORKERS_PER_HOST),
    description="Workers/host mode:"
)
fixed_workers_in = widgets.BoundedIntText(value=max(1, MIN_WORKERS_PER_HOST), min=1, max=512, step=1, description="Fixed workers/host:")
cpu_reserve_in = widgets.BoundedIntText(value=int(CPU_RESERVE), min=0, max=16, step=1, description="CPU reserve:")
min_workers_in = widgets.BoundedIntText(value=int(MIN_WORKERS_PER_HOST), min=1, max=512, step=1, description="Min workers/host:")
max_workers_in = widgets.BoundedIntText(value=int(MAX_WORKERS_PER_HOST), min=1, max=1024, step=1, description="Max workers/host:")

def _toggle_fixed_inputs(_=None):
    # Always visible; disable when mode=auto (do not hide)
    fixed_workers_in.disabled = (workers_per_host_mode_in.value != "fixed")
_toggle_fixed_inputs()
workers_per_host_mode_in.observe(_toggle_fixed_inputs, names="value")

# Small loading labels
shape_loading_lbl = widgets.HTML("")
try:
    shape_loading_lbl.add_class("loading-note")
except Exception:
    shape_loading_lbl.value = "<small style='color:#555'> </small>"
img_loading_lbl   = widgets.HTML("")
try:
    img_loading_lbl.add_class("loading-note")
except Exception:
    img_loading_lbl.value = "<small style='color:#555'> </small>"

# Helpers to enforce label non-truncation and consistent widths
def _fit(*controls, width="100%"):
    for c in controls:
        try:
            c.style.description_width = "initial"
            c.layout.width = width
        except Exception:
            pass

# Pre-apply to static controls
_fit(region_dd, ssh_pub_dd, ssh_priv_dd, gen_count_in, shape_filter,
     enable_v6_cb, ssh_v4_cidr_in, ssh_v6_cidr_in,
     test_mode_dd, wait_time_in, conn_to_in, read_to_in, verify_tls_in, ui_port_in,
     workers_per_host_mode_in, fixed_workers_in, cpu_reserve_in, min_workers_in, max_workers_in,
     targets_in, width="100%")

# Centered Apply button
apply_btn = widgets.Button(description="Apply", button_style="primary", icon="check",
                           layout=widgets.Layout(width="240px", height="36px", align_self="center"))
apply_center = widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center"))

def rebuild_dynamic(_=None):
    err_out.clear_output()
    issues_strip.value = ""
    try:
        idc = identity_client_for_current()
        cc  = compute_client_for_current()

        # Location
        comp_opts = list_compartments(idc)
        comp_dd = widgets.Dropdown(options=comp_opts, value=comp_opts[0][1], description="Compartment:", layout=widgets.Layout(width="100%"))
        ad_names = list_ads(idc)
        ad_dd = widgets.Dropdown(options=[(n, n) for n in ad_names], value=ad_names[0], description="Availability Domain (AD):", layout=widgets.Layout(width="100%"))
        _state["comp_dd"], _state["ad_dd"] = comp_dd, ad_dd

        # Shapes
        shape_loading_lbl.value = "<small>loading shapes…</small>"
        all_shapes = list_shapes(cc)
        shape_loading_lbl.value = ""
        def filtered_shapes():
            s = (shape_filter.value or "").strip().lower()
            return [n for n in all_shapes if (s in n.lower())] if s else all_shapes

        gen_shape_dd = widgets.Dropdown(
            options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")],
            value=(filtered_shapes()[0] if filtered_shapes() else ""),
            description="Generator shape:",
            layout=widgets.Layout(width="100%")
        )
        _state["gen_shape_dd"] = gen_shape_dd

        _ensure_flex_inputs()
        _toggle_flex(gen_shape_dd.value)

        def on_shape_filter(_ch):
            opts = filtered_shapes()
            gen_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            gen_shape_dd.value   = opts[0] if opts else ""
            _toggle_flex(gen_shape_dd.value)
            refresh_images()
            _err(lbl_err_shape, "")

        shape_filter.observe(on_shape_filter, names="value")
        gen_shape_dd.observe(lambda ch: (_toggle_flex(gen_shape_dd.value), refresh_images(), _err(lbl_err_shape, "")), names="value")

        # Images
        image_dd = widgets.Dropdown(options=[("Select compartment and shape first", "")], value="", description="Image:", layout=widgets.Layout(width="100%"))
        _state["image_dd"] = image_dd

        def refresh_images():
            try:
                img_loading_lbl.value = "<small>loading images…</small>"
                comp_id = comp_dd.value
                gshape  = gen_shape_dd.value or ""
                if not gshape:
                    image_dd.options = [("Select a shape to list Oracle Linux images", "")]
                    image_dd.value = ""
                    img_loading_lbl.value = ""
                    return
                imgs = oci.pagination.list_call_get_all_results(
                    cc.list_images, comp_id,
                    operating_system="Oracle Linux", sort_by="TIMECREATED", sort_order="DESC",
                    shape=gshape
                ).data
                items = []
                for im in imgs[:150]:
                    try:
                        created = im.time_created.strftime("%Y-%m-%d")
                    except Exception:
                        created = ""
                    items.append((f"{im.display_name} — {im.operating_system} {im.operating_system_version} — {created}", im.id))
                if items:
                    image_dd.options = items
                    image_dd.value   = items[0][1]
                    _err(lbl_err_image, "")
                else:
                    image_dd.options = [("No compatible Oracle Linux images for selected shape/compartment/region", "")]
                    image_dd.value   = ""
                img_loading_lbl.value = ""
            except Exception:
                image_dd.options = [("Discovery error; use Reload/Reset or adjust filters", "")]
                image_dd.value   = ""
                img_loading_lbl.value = ""

        comp_dd.observe(lambda ch: (refresh_images(), _err(lbl_err_image, "")), names="value")
        refresh_images()

        # Apply anti-truncation styling and widths to dynamic widgets
        _fit(comp_dd, ad_dd, gen_shape_dd, image_dd, _state["gen_ocpus_in"], _state["gen_mem_in"], width="100%")

        # Layout sections with consistent widths per row
        loc   = section("Location", [
            row(region_dd, reload_btn, reset_btn, widths=["70%", "15%", "15%"]),
            row(comp_dd),
            row(ad_dd)
        ])

        gens  = section("Generators", [
            row(gen_count_in), lbl_err_gen_count,
            row(shape_filter, shape_loading_lbl, widths=["85%", "15%"]),
            row(gen_shape_dd), lbl_err_shape,
            row(_state["gen_ocpus_in"], _state["gen_mem_in"], widths=["50%", "50%"]),
            lbl_err_flex,
            row(image_dd, img_loading_lbl, widths=["85%", "15%"]),
            lbl_err_image
        ])

        sshs  = section("SSH Keys", [
            row(ssh_pub_dd),
            lbl_err_ssh_pub,
            row(ssh_priv_dd),
            lbl_err_ssh_priv
        ])

        ipv6s = section("IPv6", [
            row(enable_v6_cb),
            ssh_ipv6_pinned_lbl,
            row(ssh_v4_cidr_in),
            row(ssh_v6_cidr_in)
        ])

        lcfg  = section("Locust & UI", [
            row(test_mode_dd),
            row(wait_time_in, conn_to_in, read_to_in, verify_tls_in, widths=["25%", "25%", "25%", "25%"]),
            row(ui_port_in)
        ])

        wpcy  = section("Workers per host", [
            row(workers_per_host_mode_in, fixed_workers_in, widths=["50%", "50%"]),
            row(cpu_reserve_in, min_workers_in, max_workers_in, widths=["33%", "33%", "34%"]),
            lbl_err_workers
        ])

        tsec  = section("External Targets", [
            targets_in,
            targets_preview,
            targets_hint
        ])

        actions = section("Actions", [
            issues_strip,
            apply_center,
            err_out,
            summary_out
        ])

        dynamic_box.children = [loc, gens, sshs, ipv6s, lcfg, wpcy, tsec, actions]
        err_out.clear_output()

    except Exception as e:
        dynamic_box.children = []
        with err_out:
            print("[error] UI rebuild failed:", repr(e))

def on_apply_clicked(_):
    # Always-enabled Apply; collect validation issues and render inline + compact strip
    try:
        # Global declarations must come first (before any use/assignment)
        global SELECTED_REGION, REGION, COMPARTMENT_ID, AD_A, IMAGE_ID
        global GENERATOR_COUNT, GENERATOR_SHAPE, GENERATOR_OCPUS, GENERATOR_MEMORY_GB
        global SSH_PUBLIC_KEY_PATH, SSH_PRIVATE_KEY_PATH
        global ENABLE_IPV6_ON_GENERATORS, USE_SSH_IPV6, SSH_ALLOWED_CIDR, SSH_ALLOWED_V6_CIDR
        global TEST_MODE, LOCUST_WAIT_TIME_SEC, LOCUST_CONNECT_TIMEOUT, LOCUST_READ_TIMEOUT, LOCUST_VERIFY_TLS, UI_WEB_PORT
        global EXTERNAL_TARGETS_TEXT
        global WORKERS_PER_HOST, CPU_RESERVE, MIN_WORKERS_PER_HOST, MAX_WORKERS_PER_HOST

        issues = []

        # Clear previous errors
        for lbl in (lbl_err_shape, lbl_err_image, lbl_err_gen_count, lbl_err_ssh_pub, lbl_err_ssh_priv, lbl_err_flex, lbl_err_workers):
            _err(lbl, "")
        issues_strip.value = ""
        err_out.clear_output()
        summary_out.clear_output()

        # Read current widgets
        cc = compute_client_for_current()
        SELECTED_REGION = region_dd.value
        comp_dd = _state["comp_dd"]
        ad_dd   = _state["ad_dd"]
        gen_shape_dd = _state["gen_shape_dd"]
        image_dd     = _state["image_dd"]
        _ensure_flex_inputs()

        # Basic presence checks
        if not gen_shape_dd.value:
            _err(lbl_err_shape, "Select a generator shape.")
            issues.append("shape")

        if not image_dd.value:
            _err(lbl_err_image, "Select an Oracle Linux image (list may be empty for this shape/compartment/region).")
            issues.append("image")

        if int(gen_count_in.value) < 1:
            _err(lbl_err_gen_count, "Generators must be ≥ 1.")
            issues.append("generators")

        # SSH key existence
        sel_pub  = os.path.expanduser(ssh_pub_dd.value or SSH_PUBLIC_KEY_PATH)
        sel_priv = os.path.expanduser(ssh_priv_dd.value or SSH_PRIVATE_KEY_PATH)
        if not sel_pub or not os.path.exists(sel_pub):
            _err(lbl_err_ssh_pub, "SSH public key file not found.")
            issues.append("ssh_pub")
        if not sel_priv or not os.path.exists(sel_priv):
            _err(lbl_err_ssh_priv, "SSH private key file not found.")
            issues.append("ssh_priv")

        # Flex-only numeric checks
        if (gen_shape_dd.value or "").endswith(".Flex"):
            try:
                oc = int(_state["gen_ocpus_in"].value)
                mg = int(_state["gen_mem_in"].value)
                if oc < 1 or mg < 1:
                    raise ValueError
            except Exception:
                _err(lbl_err_flex, "OCPUs/Memory must be ≥ 1 for Flex shapes.")
                issues.append("flex_ocpus_mem")

        # Workers policy checks
        try:
            mn = int(min_workers_in.value)
            mx = int(max_workers_in.value)
            if mn > mx:
                _err(lbl_err_workers, "Min workers/host cannot exceed Max workers/host.")
                issues.append("workers_min_max")
        except Exception:
            _err(lbl_err_workers, "Workers/host values must be valid integers.")
            issues.append("workers_values")

        # If any critical issues -> show compact strip and stop (do not persist env)
        if issues:
            # If issues_strip had inline fallback earlier, ensure it's visible here
            if "issues-strip" not in (issues_strip.value or ""):
                issues_strip.value = "<div class='issues-strip'><b>Issues:</b> " + ", ".join(sorted(set(issues))) + "</div>"
            else:
                issues_strip.value = "<b>Issues:</b> " + ", ".join(sorted(set(issues)))
            with err_out:
                print("Apply aborted due to issues listed above.")
            return

        # Persist non-secret env for downstream cells (same behavior as original)
        REGION = SELECTED_REGION
        COMPARTMENT_ID = comp_dd.value
        AD_A           = ad_dd.value

        GENERATOR_SHAPE = gen_shape_dd.value
        GENERATOR_COUNT = int(gen_count_in.value)
        if GENERATOR_SHAPE.endswith(".Flex"):
            GENERATOR_OCPUS     = float(_state["gen_ocpus_in"].value)
            GENERATOR_MEMORY_GB = float(_state["gen_mem_in"].value)

        IMAGE_ID = image_dd.value or ""

        SSH_PUBLIC_KEY_PATH  = os.path.expanduser(ssh_pub_dd.value or SSH_PUBLIC_KEY_PATH)
        SSH_PRIVATE_KEY_PATH = os.path.expanduser(ssh_priv_dd.value or SSH_PRIVATE_KEY_PATH)

        ENABLE_IPV6_ON_GENERATORS = bool(enable_v6_cb.value)
        USE_SSH_IPV6              = False  # pinned to IPv4

        SSH_ALLOWED_CIDR    = (ssh_v4_cidr_in.value or "0.0.0.0/0").strip()
        SSH_ALLOWED_V6_CIDR = (ssh_v6_cidr_in.value or "::/0").strip()

        TEST_MODE              = test_mode_dd.value
        LOCUST_WAIT_TIME_SEC   = float(wait_time_in.value)
        LOCUST_CONNECT_TIMEOUT = int(conn_to_in.value)
        LOCUST_READ_TIMEOUT    = int(read_to_in.value)
        LOCUST_VERIFY_TLS      = bool(verify_tls_in.value)
        UI_WEB_PORT            = int(ui_port_in.value)

        EXTERNAL_TARGETS_TEXT = (targets_in.value or "").strip()
        norm_targets = _normalize_targets(EXTERNAL_TARGETS_TEXT)

        mode = (workers_per_host_mode_in.value or "auto").strip().lower()
        if mode == "auto":
            WORKERS_PER_HOST = "auto"
        else:
            WORKERS_PER_HOST = int(fixed_workers_in.value)
        CPU_RESERVE          = int(cpu_reserve_in.value)
        MIN_WORKERS_PER_HOST = int(min_workers_in.value)
        MAX_WORKERS_PER_HOST = int(max_workers_in.value)

        os.environ["OCI_CONFIG_FILE"] = OCI_CONFIG_FILE
        os.environ["OCI_PROFILE"]     = OCI_PROFILE
        os.environ["REGION"]          = REGION
        os.environ["COMPARTMENT_ID"]  = COMPARTMENT_ID
        os.environ["AD_A"]            = AD_A
        os.environ["IMAGE_ID"]        = IMAGE_ID

        os.environ["GENERATOR_COUNT"]     = str(GENERATOR_COUNT)
        os.environ["GENERATOR_SHAPE"]     = GENERATOR_SHAPE
        os.environ["GENERATOR_OCPUS"]     = str(GENERATOR_OCPUS)
        os.environ["GENERATOR_MEMORY_GB"] = str(GENERATOR_MEMORY_GB)

        os.environ["SSH_PUBLIC_KEY_PATH"]  = SSH_PUBLIC_KEY_PATH
        os.environ["SSH_PRIVATE_KEY_PATH"] = SSH_PRIVATE_KEY_PATH

        os.environ["ENABLE_IPV6_ON_GENERATORS"] = "true" if ENABLE_IPV6_ON_GENERATORS else "false"
        os.environ["USE_SSH_IPV6"]              = "false"  # pinned to IPv4
        os.environ["SSH_ALLOWED_CIDR"]          = SSH_ALLOWED_CIDR
        os.environ["SSH_ALLOWED_V6_CIDR"]       = SSH_ALLOWED_V6_CIDR

        os.environ["TEST_MODE"] = TEST_MODE
        os.environ["LOCUST_WAIT_TIME_SEC"]      = str(LOCUST_WAIT_TIME_SEC)
        os.environ["LOCUST_CONNECT_TIMEOUT_MS"] = str(LOCUST_CONNECT_TIMEOUT)
        os.environ["LOCUST_READ_TIMEOUT_MS"]    = str(LOCUST_READ_TIMEOUT)
        os.environ["LOCUST_VERIFY_TLS"]         = "true" if LOCUST_VERIFY_TLS else "false"
        os.environ["UI_WEB_PORT"]               = str(UI_WEB_PORT)

        os.environ["EXTERNAL_TARGETS_TEXT"] = ",".join(norm_targets)

        os.environ["WORKERS_PER_HOST"]     = str(WORKERS_PER_HOST)  # "auto" or numeric as string
        os.environ["CPU_RESERVE"]          = str(CPU_RESERVE)
        os.environ["MIN_WORKERS_PER_HOST"] = str(MIN_WORKERS_PER_HOST)
        os.environ["MAX_WORKERS_PER_HOST"] = str(MAX_WORKERS_PER_HOST)

        def _bn(p):
            try:
                return os.path.basename(p) if p else "(unset)"
            except Exception:
                return "(unset)"

        # Summary (concise but complete)
        summary_out.clear_output()
        with summary_out:
            display(HTML(f"""
<div style="border:{BORDER};background:{SECTION_BG};padding:{PAD};">
  <b>Selections applied</b>
  <div style="font-family:ui-monospace; white-space:pre-wrap;">
Region={REGION}
Compartment={COMPARTMENT_ID}
AD={AD_A}

Generators: count={GENERATOR_COUNT} | shape={GENERATOR_SHAPE} | OCPUs={(GENERATOR_OCPUS if GENERATOR_SHAPE.endswith('.Flex') else '-')} | MemGB={(GENERATOR_MEMORY_GB if GENERATOR_SHAPE.endswith('.Flex') else '-')}
Image: {(IMAGE_ID if not IMAGE_ID or len(IMAGE_ID) < 20 else '…' + IMAGE_ID[-16:])}

IPv6 assignment: {ENABLE_IPV6_ON_GENERATORS} | SSH over IPv6: false (pinned to IPv4)
SSH allowed v4: {SSH_ALLOWED_CIDR}
SSH allowed v6: {SSH_ALLOWED_V6_CIDR}

Locust: mode={TEST_MODE} | wait(s)={LOCUST_WAIT_TIME_SEC} | connect(ms)={LOCUST_CONNECT_TIMEOUT} | read(ms)={LOCUST_READ_TIMEOUT} | verify_tls={LOCUST_VERIFY_TLS}
UI: port={UI_WEB_PORT}

CPU policy: mode={'auto' if isinstance(WORKERS_PER_HOST, str) and WORKERS_PER_HOST=='auto' else 'fixed'} | fixed={WORKERS_PER_HOST if not (isinstance(WORKERS_PER_HOST, str) and WORKERS_PER_HOST=='auto') else '-'} | reserve={CPU_RESERVE} | min={MIN_WORKERS_PER_HOST} | max={MAX_WORKERS_PER_HOST}

External targets: {len(norm_targets)} {(('=> ' + ', '.join(norm_targets[:2])) if norm_targets else '')}

SSH key (pub): {_bn(SSH_PUBLIC_KEY_PATH)}
SSH key (priv): {_bn(SSH_PRIVATE_KEY_PATH)}
  </div>
</div>
"""))
        print("Cell 3 complete: Selections applied. NEXT: Run Cell 4 (cloud-init), Cell 5 (Terraform main.tf), then Cell 6 (terraform.tfvars).")

    except Exception as e:
        with err_out:
            print("[error] Apply failed:", repr(e))

# Wire up buttons and events
apply_btn.on_click(on_apply_clicked)
reload_btn.on_click(lambda _: rebuild_dynamic())
reset_btn.on_click(lambda _: (setattr(region_dd, "value", default_region), rebuild_dynamic()))
region_dd.observe(rebuild_dynamic, names="value")

# Initial UI build and display
display(_css)
rebuild_dynamic()
container = widgets.VBox([dynamic_box], layout=widgets.Layout(width="100%", max_width=CONTAINER_MAX_W, margin="0 auto"))
display(container)
print("Cell 3 loaded: Configure settings (Flex inputs appear only when shape is .Flex), CPU policy, and click Apply.")


In [ ]:
# Cell 4 — Objective: Write generator cloud-init (tuning-only; no package installs). Create workspace + locustfile.
# Notes:
# - Do NOT run dnf/yum/pip in cloud-init. First boot remains deterministic.
# - The actual locust installation will be performed later via an SSH-based prepare step.

import os

os.makedirs("cloud-init", exist_ok=True)

generator_cloud_init = r"""#!/bin/bash
set -euo pipefail

# Stop/disable firewalld (harmless if not present)
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi

# Enable Oracle Cloud Agent if present
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi

# Kernel tuning for high connection rates (safe defaults)
cat <<EOF >/etc/sysctl.d/99-freewheel.conf
net.core.somaxconn=65535
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
fs.file-max=1000000
EOF
sysctl --system || true
echo "* - nofile 1048576" >> /etc/security/limits.conf || true

# Create workspace (no package installs here)
mkdir -p /home/opc/locustwork/results
chown -R opc:opc /home/opc/locustwork

# Multi-capable locustfile (uses environment for targets/paths/mode; safe with UI/prepare flow)
cat > /home/opc/locustwork/locustfile.py <<'PY'
import os
from itertools import cycle
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
RAW_TARGETS       = os.environ.get("LOCUST_TARGETS", "")
TARGETS           = [t.strip() for t in RAW_TARGETS.split(",") if t.strip()]
NAME_BY_VIP       = os.environ.get("LOCUST_NAME_BY_VIP", "true").lower() == "true"

class CpsUser(HttpUser):
    host = os.environ.get("LOCUST_DEFAULT_HOST", "https://127.0.0.1")
    wait_time = constant(WAIT_TIME_S)

    def on_start(self):
        self._targets = cycle(TARGETS) if TARGETS else None

    def _next_base(self):
        try:
            return next(self._targets) if self._targets else (self.host or "")
        except Exception:
            return self.host or ""

    @task
    def do_request(self):
        path = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        base = self._next_base()
        url  = f"{base}{path}" if base else path
        headers = {"Connection": "close"} if MODE == "cps" else {}
        name_val = path
        if NAME_BY_VIP and base:
            vip = base.replace("https://","").replace("http://","")
            name_val = f"{vip}{path}"
        self.client.get(url, headers=headers,
                        verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=name_val)
PY

chown -R opc:opc /home/opc/locustwork
echo READY
"""

with open("cloud-init/generator.sh", "w") as f:
    f.write(generator_cloud_init)

print("Cell 4 complete: Wrote cloud-init/generator.sh (tuning-only; workspace + locustfile; no dnf/pip).")
print("NEXT: Run Cell 5 to generate Terraform (generators-only infra with public egress).")


In [ ]:
# Cell 5 — Objective: Terraform (generators-only; public egress) with ALL protocols allowed stateless (NSG + Default Security List)
# - Default Security List: protocol = "all" stateless egress to 0.0.0.0/0 and ::/0, and protocol = "all" stateless ingress from 0.0.0.0/0 and ::/0.
# - NSG attached to instances: same "all/all" stateless rules so NSG does not constrain any protocol/port.
# - This is the only generic stateless model that guarantees return traffic for arbitrary protocols/ports without enumerating each.

import os

gen_shape_config_block = ""
if (GENERATOR_SHAPE or "").endswith(".Flex"):
    gen_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (GENERATOR_OCPUS, GENERATOR_MEMORY_GB)

terraform_config_template = """terraform {
  required_providers {
    oci = {
      source  = "oracle/oci"
      version = ">= 7.0.0"
    }
  }
}

provider "oci" {
  config_file_profile = var.oci_profile
  region              = var.region
}

# Variables
variable "oci_profile" {}
variable "region" {}
variable "compartment_id" {}
variable "ad_a" {}
variable "image_id" {}
variable "ssh_public_key_content" {}
variable "ssh_private_key_path" {}

variable "generator_count" {}
variable "generator_shape" {}
variable "generator_ocpus" {
  type    = number
  default = 8
}
variable "generator_memory_gb" {
  type    = number
  default = 32
}

variable "enable_ipv6_on_generators" {
  type    = bool
  default = false
}

variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}

# VCN (dual-stack)
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.20.0.0/16"
  compartment_id = var.compartment_id
  display_name   = "gens-only-vcn"
  is_ipv6enabled = true
}

locals {
  vcn_ipv6_base = oci_core_vcn.vcn.ipv6cidr_blocks[0]
  gens_pub_ipv6 = cidrsubnet(local.vcn_ipv6_base, 8, 1)
}

resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "gens-only-igw"
}

resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "gens-only-rt-public"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }

  route_rules {
    destination       = "::/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}

resource "oci_core_subnet" "gens_pub" {
  cidr_block                 = "10.20.1.0/24"
  ipv6cidr_block             = local.gens_pub_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "gens-only-public-subnet"
  route_table_id             = oci_core_route_table.rt_public.id
  prohibit_public_ip_on_vnic = false
}

# Default Security List — ALL protocols allowed stateless (egress and ingress), v4 and v6
resource "oci_core_default_security_list" "vcn_default" {
  manage_default_resource_id = oci_core_vcn.vcn.default_security_list_id

  # EGRESS: protocol "all" (IPv4 + IPv6), stateless
  egress_security_rules {
    protocol         = "all"
    destination      = "0.0.0.0/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }
  egress_security_rules {
    protocol         = "all"
    destination      = "::/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  # INGRESS: protocol "all" (IPv4 + IPv6), stateless — required so return traffic is not blocked
  ingress_security_rules {
    protocol    = "all"
    source      = "0.0.0.0/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
  ingress_security_rules {
    protocol    = "all"
    source      = "::/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
}

# NSG — also ALL protocols allowed stateless (egress and ingress), so NSG does not constrain traffic
resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-generators"
}

resource "oci_core_network_security_group_security_rule" "nsg_egress_all_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_egress_all_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_ingress_all_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_ingress_all_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}

# Generator instances
resource "oci_core_instance" "generator" {
  count               = var.generator_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape
__GEN_SHAPE_CONFIG__

  source_details {
    source_type = "image"
    source_id   = var.image_id
  }

  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [oci_core_network_security_group.nsg_generators.id]
  }

  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false

    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }

  display_name = "ext-gen-${count.index}"

  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = filebase64("${path.module}/cloud-init/generator.sh")
  }
}

# Optional IPv6 assignment
data "oci_core_vnic_attachments" "gen_vnic" {
  count          = var.enable_ipv6_on_generators ? var.generator_count : 0
  compartment_id = var.compartment_id
  instance_id    = oci_core_instance.generator[count.index].id
}

resource "oci_core_ipv6" "gen_v6" {
  count     = var.enable_ipv6_on_generators ? var.generator_count : 0
  subnet_id = oci_core_subnet.gens_pub.id
  vnic_id   = data.oci_core_vnic_attachments.gen_vnic[count.index].vnic_attachments[0].vnic_id
}

# Outputs
output "gen_public_ips" {
  value = [for i in oci_core_instance.generator : i.public_ip]
}

output "gen_ipv6_ips" {
  value = oci_core_ipv6.gen_v6[*].ip_address
}
"""

terraform_config = terraform_config_template.replace("__GEN_SHAPE_CONFIG__", gen_shape_config_block)

with open("main.tf", "w") as f:
    f.write(terraform_config)

print("Cell 5 complete: Terraform main.tf updated — ALL protocols allowed stateless (NSG + Default Security List).")
print("WARNING: This fully opens inbound and outbound at NSG and default SL (stateless). Intended for general-purpose generators.")
print("NEXT: Destroy previous infra if present, then run Cell 7 (init/apply), Cell 8 (capture outputs), Cell 10 (optional triage).")


In [ ]:
# Cell 6 — Objective: Write terraform.tfvars for the all‑protocols, stateless security model
# - Validates selections from Cell 3; writes tfvars matching the updated main.tf (Cell 5).
# - Variables included: oci_profile, region, compartment_id, ad_a, image_id,
#   ssh_public_key_content, ssh_private_key_path, generator_* and enable_ipv6_on_generators.
# - No secrets printed.

import os

def _fail(msg: str):
    raise ValueError(msg)

# Required selections from Cell 3 (persisted to env)
OCI_PROFILE    = os.environ.get("OCI_PROFILE", "DEFAULT")
REGION         = os.environ.get("REGION", "")
COMPARTMENT_ID = os.environ.get("COMPARTMENT_ID", "")
AD_A           = os.environ.get("AD_A", "")
IMAGE_ID       = os.environ.get("IMAGE_ID", "")

GENERATOR_COUNT     = os.environ.get("GENERATOR_COUNT", "")
GENERATOR_SHAPE     = os.environ.get("GENERATOR_SHAPE", "")
GENERATOR_OCPUS     = os.environ.get("GENERATOR_OCPUS", "")
GENERATOR_MEMORY_GB = os.environ.get("GENERATOR_MEMORY_GB", "")

SSH_PUBLIC_KEY_PATH  = os.environ.get("SSH_PUBLIC_KEY_PATH", "")
SSH_PRIVATE_KEY_PATH = os.environ.get("SSH_PRIVATE_KEY_PATH", "")

ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")

# Basic validations
if not REGION:
    _fail("REGION is not set.")
if not COMPARTMENT_ID:
    _fail("COMPARTMENT_ID is not set. Run Cell 3 and Apply.")
if not AD_A:
    _fail("AD_A is not set. Run Cell 3 and Apply.")
if not IMAGE_ID:
    _fail("IMAGE_ID is not set. Run Cell 3 and select an Image.")
if not GENERATOR_SHAPE:
    _fail("GENERATOR_SHAPE is not set. Run Cell 3 and select a Generator shape.")

try:
    _ = int(GENERATOR_COUNT)
except Exception:
    _fail("GENERATOR_COUNT is invalid. Re-apply Cell 3 with a valid integer.")

if GENERATOR_SHAPE.endswith(".Flex"):
    try:
        _ = float(GENERATOR_OCPUS)
        _ = float(GENERATOR_MEMORY_GB)
    except Exception:
        _fail("For Flex shapes, GENERATOR_OCPUS/GENERATOR_MEMORY_GB must be valid numbers. Re-apply Cell 3.")
else:
    # For non-Flex shapes, these are ignored in main.tf; set to 0 explicitly
    GENERATOR_OCPUS = "0"
    GENERATOR_MEMORY_GB = "0"

if not SSH_PUBLIC_KEY_PATH or not os.path.exists(os.path.expanduser(SSH_PUBLIC_KEY_PATH)):
    _fail(f"SSH public key missing. Pick a valid key in Cell 3. Current: {SSH_PUBLIC_KEY_PATH}")
if not SSH_PRIVATE_KEY_PATH or not os.path.exists(os.path.expanduser(SSH_PRIVATE_KEY_PATH)):
    _fail(f"SSH private key missing. Pick a valid key in Cell 3. Current: {SSH_PRIVATE_KEY_PATH}")

# Load SSH public key content (string for tfvars)
with open(os.path.expanduser(SSH_PUBLIC_KEY_PATH), "r") as f:
    ssh_public_key_content = (f.read() or "").strip()
if not ssh_public_key_content:
    _fail("SSH public key file is empty. Provide a valid public key.")

ssh_public_key_content_escaped = ssh_public_key_content.replace('"', '\\"')
ssh_private_key_abs = os.path.expanduser(SSH_PRIVATE_KEY_PATH)

# Write tfvars (match updated main.tf variables exactly)
tfvars = f"""
oci_profile            = "{OCI_PROFILE}"
region                 = "{REGION}"
compartment_id         = "{COMPARTMENT_ID}"
ad_a                   = "{AD_A}"
image_id               = "{IMAGE_ID}"
ssh_public_key_content = "{ssh_public_key_content_escaped}"
ssh_private_key_path   = "{ssh_private_key_abs}"

generator_count     = {int(GENERATOR_COUNT)}
generator_shape     = "{GENERATOR_SHAPE}"
generator_ocpus     = {int(float(GENERATOR_OCPUS)) if GENERATOR_SHAPE.endswith(".Flex") else 0}
generator_memory_gb = {int(float(GENERATOR_MEMORY_GB)) if GENERATOR_SHAPE.endswith(".Flex") else 0}

enable_ipv6_on_generators = {"true" if ENABLE_IPV6_ON_GENERATORS else "false"}
""".lstrip()

with open("terraform.tfvars", "w") as f:
    f.write(tfvars)

# Print a concise summary (no secrets)
print("Cell 6 complete: terraform.tfvars written for all‑protocols stateless model.")
print(f"  region={REGION}")
print(f"  compartment_id={COMPARTMENT_ID}")
print(f"  ad_a={AD_A}")
print(f"  generator_shape={GENERATOR_SHAPE} | count={GENERATOR_COUNT} | flex_ocpus={GENERATOR_OCPUS if GENERATOR_SHAPE.endswith('.Flex') else '-'} | flex_mem_gb={GENERATOR_MEMORY_GB if GENERATOR_SHAPE.endswith('.Flex') else '-'}")
print(f"  enable_ipv6_on_generators={ENABLE_IPV6_ON_GENERATORS}")
print("NEXT: Destroy previous infra if present, then run Cell 7 (terraform init/apply).")


In [ ]:
# Cell 7 — Objective: Initialize and apply Terraform (all‑stateless security model)
# - Assumes you destroyed any previous deployment, and updated main.tf (Cell 5) and terraform.tfvars (Cell 6).
# - Runs terraform init and a non-interactive apply using the provided tfvars.
# - No changes are needed here for the stateless model beyond re-applying.

# Optional: quick check for terraform binary (informative)
try:
    import shutil
    if shutil.which("terraform") is None:
        print("Warning: 'terraform' not found in PATH. Install Terraform and re-run this cell.")
except Exception:
    pass

# Initialize and apply with the updated configuration
!terraform init
!terraform apply -auto-approve -var-file=terraform.tfvars

print("Cell 7 complete: Terraform apply finished for the all-stateless model.")
print("NEXT: Run Cell 8 to capture generator IPv4/IPv6 outputs, then Cell 9 to confirm outbound over IPv4/IPv6.")


In [ ]:
# Cell 8 — Objective: Capture generator outputs (public IPv4/IPv6), persist to environment, and write artifacts
# - Reads terraform outputs (JSON) for gen_public_ips and gen_ipv6_ips.
# - Persists GEN_IPS_V4_JSON and GEN_IPS_V6_JSON to the notebook environment for later cells.
# - Creates a paired map (index-wise) if both IPv4 and IPv6 are present.
# - Writes snapshot and text files under OUTPUT_DIR with the current TS_UTC (created if missing).

import os, json, subprocess
from datetime import datetime, timezone
from pathlib import Path

def _tf_output_json() -> dict:
    r = subprocess.run(["terraform", "output", "-json"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"terraform output failed:\n{r.stderr}")
    try:
        return json.loads(r.stdout or "{}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Could not parse terraform output JSON: {e}")

def _val(outputs: dict, key: str):
    obj = outputs.get(key, {})
    if isinstance(obj, dict) and "value" in obj:
        return obj.get("value")
    return obj if obj is not None else None

def _ensure_ts_env() -> str:
    ts_env = os.environ.get("TS_UTC", "").strip()
    if ts_env:
        return ts_env
    ts_new = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    os.environ["TS_UTC"] = ts_new
    return ts_new

def _write_lines(fp: Path, lines: list[str]):
    fp.parent.mkdir(parents=True, exist_ok=True)
    with fp.open("w") as f:
        for ln in lines:
            f.write(str(ln).strip() + "\n")

# Load outputs from Terraform
outputs = _tf_output_json()
GEN_IPS_V4 = _val(outputs, "gen_public_ips") or []
GEN_IPS_V6 = _val(outputs, "gen_ipv6_ips") or []

# Persist to environment for downstream cells
os.environ["GEN_IPS_V4_JSON"] = json.dumps(GEN_IPS_V4)
os.environ["GEN_IPS_V6_JSON"] = json.dumps(GEN_IPS_V6)

# Build paired map (index-based) if both are present
pairs = []
if GEN_IPS_V4 and GEN_IPS_V6:
    m = min(len(GEN_IPS_V4), len(GEN_IPS_V6))
    if m > 0:
        pairs = [{"ipv4": GEN_IPS_V4[i], "ipv6": GEN_IPS_V6[i]} for i in range(m)]
        os.environ["GEN_V4_V6_MAP_JSON"] = json.dumps(pairs)

# Artifacts
OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
TS_UTC = _ensure_ts_env()
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

snap = {
    "gen_public_ips_v4": GEN_IPS_V4,
    "gen_ipv6_ips": GEN_IPS_V6,
    "paired_v4_v6": pairs,
    "ts_utc": TS_UTC,
}
snap_path = out_dir / f"terraform_generators_{TS_UTC}.json"
with snap_path.open("w") as f:
    json.dump(snap, f, indent=2)

_write_lines(out_dir / f"gens_ipv4_{TS_UTC}.txt", GEN_IPS_V4)
_write_lines(out_dir / f"gens_ipv6_{TS_UTC}.txt", GEN_IPS_V6)

# Summary to console
print("Generators (IPv4):", GEN_IPS_V4)
print("Generators (IPv6):", GEN_IPS_V6)
if pairs:
    print(f"Paired (by index): {len(pairs)} entries")

print("Saved outputs snapshot:", str(snap_path))
print("Wrote artifacts:",
      str(out_dir / f"gens_ipv4_{TS_UTC}.txt"), ",",
      str(out_dir / f"gens_ipv6_{TS_UTC}.txt"))

print("\nCell 8 complete: Outputs captured and artifacts written.")
print("NEXT: Run Cell 9 to confirm outbound over IPv4/IPv6 from a generator (SSH over IPv4).")


In [ ]:
# Cell 9 — Objective: Confirm outbound over IPv4/IPv6 from one generator (SSH pinned to IPv4)
# - Uses the first generator IPv4 (from Cell 8 outputs).
# - Checks DNS A/AAAA and curl -4/-6 to PyPI and the regional OCI yum mirror.
# - Writes a brief report file and prints a concise summary.

import os, json, paramiko
from pathlib import Path
from datetime import datetime, timezone
from typing import Tuple

def _load_list_env(name: str, default="[]"):
    try:
        v = json.loads(os.environ.get(name, default) or default)
        return v if isinstance(v, list) else []
    except Exception:
        return []

GEN_IPS_V4 = _load_list_env("GEN_IPS_V4_JSON")
if not GEN_IPS_V4:
    raise RuntimeError("GEN_IPS_V4_JSON empty. Run Cell 8 to populate outputs.")
HOST = GEN_IPS_V4[0]

REGION = os.environ.get("REGION", "").strip()
if not REGION:
    raise RuntimeError("REGION is empty in env. Re-run Cells 2/3 to persist REGION.")

YUM_HOST   = f"yum.{REGION}.oci.oraclecloud.com"
YUM_REPOMD = f"https://{YUM_HOST}/repo/OracleLinux/OL9/ksplice/x86_64/repodata/repomd.xml"
PYPI_URL   = "https://pypi.org/simple/locust/"

TS_UTC = os.environ.get("TS_UTC") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.environ["TS_UTC"] = TS_UTC
BASE_OUT = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
OUT_DIR  = Path(BASE_OUT) / f"nettriage_{TS_UTC}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = OUT_DIR / f"{HOST.replace('.', '_')}.txt"

SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip())
if not SSH_KEY or not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not set/found. Current: {SSH_KEY or '(unset)'}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path, pw):
    for K in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return K.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key: {path}")

def ssh_exec(host: str, cmd: str, timeout: int = 40) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli  = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    full = f"bash -lc {json.dumps(cmd)}"
    _, o, e = cli.exec_command(full, timeout=timeout)
    out = (o.read().decode("utf-8", "ignore") or "").strip()
    err = (e.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = o.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

LINES = []
def add(section, cmd, timeout=40):
    LINES.append(f"\n==== {section} ====\n$ {cmd}\n")
    out, err, rc = ssh_exec(HOST, cmd, timeout=timeout)
    LINES.append(out if out else (err if err else f"(no output) rc={rc}"))

# DNS checks
add("DNS A/AAAA for pypi.org", "dig +short A pypi.org; echo '---'; dig +short AAAA pypi.org", timeout=20)
add(f"DNS A/AAAA for {YUM_HOST}", f"dig +short A {YUM_HOST}; echo '---'; dig +short AAAA {YUM_HOST}", timeout=20)

# Reachability (cap with 'timeout'; print HTTP code, IP, and total time)
cmd_pypi_v4 = f"timeout -s TERM 12s curl -4 -sS -o /dev/null -w 'pypi v4 code:%{{http_code}} ip:%{{remote_ip}} time:%{{time_total}}\\n' --connect-timeout 5 --max-time 10 {PYPI_URL} || echo curl4_failed"
cmd_pypi_v6 = f"timeout -s TERM 12s curl -6 -sS -o /dev/null -w 'pypi v6 code:%{{http_code}} ip:%{{remote_ip}} time:%{{time_total}}\\n' --connect-timeout 5 --max-time 10 {PYPI_URL} || echo curl6_failed"
cmd_yum_v4  = f"timeout -s TERM 12s curl -4 -sS -o /dev/null -w 'yum v4 code:%{{http_code}} ip:%{{remote_ip}} time:%{{time_total}}\\n' --connect-timeout 5 --max-time 10 {YUM_REPOMD} || echo curl4_failed"
cmd_yum_v6  = f"timeout -s TERM 12s curl -6 -sS -o /dev/null -w 'yum v6 code:%{{http_code}} ip:%{{remote_ip}} time:%{{time_total}}\\n' --connect-timeout 5 --max-time 10 {YUM_REPOMD} || echo curl6_failed"

add("curl -4 to PyPI", cmd_pypi_v4, timeout=20)
add("curl -6 to PyPI", cmd_pypi_v6, timeout=20)
add("curl -4 to OCI yum repomd", cmd_yum_v4, timeout=20)
add("curl -6 to OCI yum repomd", cmd_yum_v6, timeout=20)

# Routes
add("IPv4 default route check", "ip -4 route get 1.1.1.1 2>/dev/null || ip -4 route || true", timeout=10)
add("IPv6 routes", "ip -6 route || true", timeout=10)

# Write full report to file
REPORT.write_text("\n".join(LINES), encoding="utf-8")

# Short summary (safe command wrapping)
sum_v4_pypi, _, _ = ssh_exec(HOST, cmd_pypi_v4, timeout=20)
sum_v6_pypi, _, _ = ssh_exec(HOST, cmd_pypi_v6, timeout=20)
sum_v4_yum,  _, _ = ssh_exec(HOST, cmd_yum_v4,  timeout=20)
sum_v6_yum,  _, _ = ssh_exec(HOST, cmd_yum_v6,  timeout=20)

print("Cell 9 complete: Outbound confirmation collected.")
print("Report:", str(REPORT))
print("Quick summary:")
print(" ", (sum_v4_pypi or "pypi v4: (no output)"))
print(" ", (sum_v6_pypi or "pypi v6: (no output)"))
print(" ", (sum_v4_yum  or "yum v4: (no output)"))
print(" ", (sum_v6_yum  or "yum v6: (no output)"))
print("NEXT: Run Cell 10 to Repair+Prepare (locust user-scope + tmux), then Cell 11 to start the Locust master and workers.")


In [ ]:
# Cell 10 — Objective: Repair + Prepare generators in parallel (IPv4 SSH pinned) + install tmux
# - Per host (sequential steps):
#   1) Ensure workspace
#   2) Ensure pip (bootstrap if missing)
#   3) Upgrade pip/setuptools/wheel (user-scope)
#   4) Install/upgrade locust (user-scope)
#   5) Install tmux via system package manager (sudo, non-interactive)
#   6) Verify locust import and version; print tmux -V
# - Across hosts: run in parallel using a thread pool (configurable via PREP_CONCURRENCY, default 5).
# - SSH is pinned to IPv4.

import os, json, paramiko, concurrent.futures, threading
from pathlib import Path
from typing import List, Tuple

# Concurrency (hosts processed at the same time)
try:
    PREP_CONCURRENCY = int(os.environ.get("PREP_CONCURRENCY", "5"))
    if PREP_CONCURRENCY < 1:
        PREP_CONCURRENCY = 1
except Exception:
    PREP_CONCURRENCY = 5

def _load_list_env(name: str, default="[]") -> List[str]:
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

# Use IPv4 for SSH (pinned)
GEN_IPS = _load_list_env("GEN_IPS_V4_JSON")
if not GEN_IPS:
    raise RuntimeError("GEN_IPS_V4_JSON is empty. Run Cell 8 to populate outputs.")

SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip())
if not SSH_KEY or not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not set or not found. Current: {SSH_KEY or '(unset)'}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path: str, pw: str | None):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key. Path: {path}")

def ssh_exec(host: str, command: str, timeout: int = 480) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8", "ignore") or "").strip()
    err = (stderr.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

# Single-line commands (PATH includes user-scope bin)
PREFIX = 'export PATH="$HOME/.local/bin:$PATH";'

CMD_MKDIR   = f'{PREFIX} mkdir -p "$HOME/locustwork/results"'
CMD_PIPCHK  = f'{PREFIX} python3 -m pip -V >/dev/null 2>&1 || (curl -fsSL --connect-timeout 10 --max-time 30 https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py && python3 /tmp/get-pip.py --user && rm -f /tmp/get-pip.py)'
CMD_BASEUP  = f'{PREFIX} python3 -m pip install --user --upgrade --no-cache-dir --timeout 40 --retries 1 pip setuptools wheel'
CMD_LOCUST  = f'{PREFIX} python3 -m pip install --user --upgrade --no-cache-dir --timeout 60 --retries 1 locust'
CMD_TMUX    = (
    f'{PREFIX} command -v tmux >/dev/null 2>&1 || '
    '( (command -v dnf >/dev/null 2>&1 && sudo -n dnf -y install tmux) || '
    '  (command -v yum >/dev/null 2>&1 && sudo -n yum -y install tmux) || '
    '  (command -v microdnf >/dev/null 2>&1 && sudo -n microdnf -y install tmux) || '
    '  (echo "tmux install failed" >&2) )'
)
CMD_VERIFY  = f'{PREFIX} python3 -c \'import importlib.util as u, importlib.metadata as md; m=u.find_spec("locust"); print("LOCUST_OK version="+md.version("locust")) if m else print("LOCUST_MISSING")\''
CMD_TMUXVER = f'{PREFIX} tmux -V || echo "tmux not installed"'

def run_host_prepare(host: str) -> Tuple[str, str]:
    logs = []
    status = "UNKNOWN"
    steps = [
        ("mkdir",   CMD_MKDIR,   60),
        ("pipchk",  CMD_PIPCHK,  180),
        ("baseup",  CMD_BASEUP,  240),
        ("locust",  CMD_LOCUST,  360),
        ("tmux",    CMD_TMUX,    240),
        ("verify",  CMD_VERIFY,   60),
        ("tmuxver", CMD_TMUXVER,  20),
    ]
    for tag, cmd, to in steps:
        out, err, rc = ssh_exec(host, cmd, timeout=to)
        logs.append(f"[{host}][{tag}][rc={rc}]\n" + (out + ("\n" if out else "")) + (("[stderr] " + err) if err else ""))
        if tag == "verify":
            line = (out or err or "").strip()
            status = line if line.startswith("LOCUST_") else (line or f"rc={rc}")
    return status, "\n".join(logs)

# Run in parallel
results = {}
logs_by_host = {}
print(f"Preparing {len(GEN_IPS)} host(s) in parallel with PREP_CONCURRENCY={PREP_CONCURRENCY} ...")
with concurrent.futures.ThreadPoolExecutor(max_workers=min(PREP_CONCURRENCY, len(GEN_IPS))) as ex:
    future_map = {ex.submit(run_host_prepare, h): h for h in GEN_IPS}
    for fut in concurrent.futures.as_completed(future_map):
        host = future_map[fut]
        try:
            status, host_logs = fut.result()
        except Exception as e:
            status, host_logs = (f"LOCUST_MISSING error={e}", f"[{host}][exception] {e}")
        results[host] = status
        logs_by_host[host] = host_logs

# Print per-host logs in a stable order
ok_hosts, bad_hosts = [], []
for h in GEN_IPS:
    print(f"\n==== Logs for {h} ====")
    print(logs_by_host.get(h, "(no logs)"))
    st = results.get(h, "UNKNOWN")
    print(f"[status] {st}")
    if st.startswith("LOCUST_OK"):
        ok_hosts.append(h)
    else:
        bad_hosts.append(h)

print("\nSummary:")
print(f"  LOCUST_OK hosts: {len(ok_hosts)}/{len(GEN_IPS)}")
if bad_hosts:
    print("  Hosts needing attention:", bad_hosts)
else:
    print("  All hosts prepared successfully (locust + tmux present).")

print("\nCell 10 complete: Parallel Repair+Prepare (locust + tmux) executed.")
print("NEXT: Run Cell 11 to start the Locust UI master and spawn workers (tmux-managed).")


In [ ]:
# Cell 11 — Objective: Start Locust UI master + workers (robust master IP, tmux-foreground, per-host nohup loop, parallel)
# - Master: runs in tmux foreground (exec) so session owns the process.
# - Workers: one SSH per host; upload start_workers.sh; nohup loop spawns all workers locally.
# - Parallel across hosts; readiness waits; robust master IP detection; SSH control-plane is IPv4.
# - Targets come from EXTERNAL_TARGETS_TEXT normalized in Cell 3; or user can set Host in the UI at runtime.

import os, json, time, shlex, paramiko, concurrent.futures
from pathlib import Path
from typing import List, Tuple

# Parallelism across hosts (hosts concurrently, not per-worker SSH)
try:
    HOST_SPAWN_CONCURRENCY = int(os.environ.get("HOST_SPAWN_CONCURRENCY", "8"))
    if HOST_SPAWN_CONCURRENCY < 1:
        HOST_SPAWN_CONCURRENCY = 1
except Exception:
    HOST_SPAWN_CONCURRENCY = 8

def _load_list_env(name: str, default="[]") -> List[str]:
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

# Resolve generator hosts (prefer IPv4; SSH is pinned to IPv4)
GEN_IPS_V4 = _load_list_env("GEN_IPS_V4_JSON")
GEN_IPS_V6 = _load_list_env("GEN_IPS_V6_JSON")
USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")
GEN_IPS = GEN_IPS_V6 if (USE_SSH_IPV6 and GEN_IPS_V6) else GEN_IPS_V4
if not GEN_IPS:
    raise RuntimeError("No generator hosts available. Run previous cells to provision and prepare generators.")
MASTER = GEN_IPS[0]

# Runtime settings (from Cell 3)
TEST_MODE                = os.environ.get("TEST_MODE", "cps").strip().lower()
LOCUST_WAIT_TIME_SEC     = os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0")
LOCUST_CONNECT_MS        = os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000")
LOCUST_READ_MS           = os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000")
LOCUST_VERIFY_TLS_BOOL   = (os.environ.get("LOCUST_VERIFY_TLS", "false").strip().lower() == "true")
HEALTH_ENDPOINT_PATH     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")
UI_WEB_PORT              = int(os.environ.get("UI_WEB_PORT", "8089"))
TARGETS_CSV              = os.environ.get("EXTERNAL_TARGETS_TEXT", "").strip()
LOCUST_WORKDIR           = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")

# CPU policy
WORKERS_PER_HOST_ENV   = os.environ.get("WORKERS_PER_HOST", "auto").strip().lower()
CPU_RESERVE            = int(os.environ.get("CPU_RESERVE", "1"))
MIN_WORKERS_PER_HOST   = int(os.environ.get("MIN_WORKERS_PER_HOST", "1"))
MAX_WORKERS_PER_HOST   = int(os.environ.get("MAX_WORKERS_PER_HOST", "32"))

def _ms_to_s_str(ms_str: str, default_s: str) -> str:
    try:
        return str(round(max(0.001, float(ms_str) / 1000.0), 3))
    except Exception:
        return default_s

CONNECT_S = _ms_to_s_str(LOCUST_CONNECT_MS, "8")
READ_S    = _ms_to_s_str(LOCUST_READ_MS, "15")

SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip() or "~/.ssh/id_rsa")
if not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not found: {SSH_KEY}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path: str, pw: str | None):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key: {path}")

def ssh_exec(host: str, command: str, timeout: int = 90) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8", "ignore") or "").strip()
    err = (stderr.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

def sftp_write(host: str, content: str, remote_path: str, mode: int = 0o755):
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    sftp = cli.open_sftp()
    try:
        parent = os.path.dirname(remote_path)
        try:
            sftp.stat(parent)
        except FileNotFoundError:
            sftp.mkdir(parent)
        with sftp.file(remote_path, "w") as f:
            f.write(content)
        sftp.chmod(remote_path, mode)
    finally:
        sftp.close(); cli.close()

# Robust master bind detection — prefer IPv4; no indexing on empty
def _detect_master_bind_and_addr() -> Tuple[str, str]:
    if USE_SSH_IPV6 and ":" in MASTER:
        return ("::", MASTER)
    # Try route 'src' first
    out, _, _ = ssh_exec(MASTER, r"ip -4 route get 1.1.1.1 2>/dev/null | awk '{for(i=1;i<=NF;i++) if($i==\"src\") {print $(i+1); exit}}'", timeout=10)
    ip = (out or "").strip()
    if not ip:
        out, _, _ = ssh_exec(MASTER, r"hostname -I 2>/dev/null | tr ' ' '\n' | awk '/^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+$/{print; exit}'", timeout=10)
        ip = (out or "").strip()
    if not ip:
        out, _, _ = ssh_exec(MASTER, r"ip -4 addr show scope global 2>/dev/null | awk '/inet /{print $2}' | cut -d/ -f1 | head -n1", timeout=10)
        ip = (out or "").strip()
    return ("0.0.0.0", (ip or MASTER))

MASTER_BIND_HOST, MASTER_ADDR_FOR_WORKERS_RAW = _detect_master_bind_and_addr()
# Last-resort sanitize to a single IPv4 token if unexpected whitespace appears
if " " in (MASTER_ADDR_FOR_WORKERS_RAW or ""):
    toks = (MASTER_ADDR_FOR_WORKERS_RAW or "").split()
    MASTER_ADDR_FOR_WORKERS_RAW = next((t for t in toks if t.count(".")==3), MASTER)
MASTER_ADDR_FOR_WORKERS = shlex.quote(MASTER_ADDR_FOR_WORKERS_RAW)

# Common environment for master/workers
def _env_exports():
    lines = [
        f"export LOCUST_MODE={shlex.quote(TEST_MODE)}",
        f"export LOCUST_HEALTH_PATH={shlex.quote(HEALTH_ENDPOINT_PATH)}",
        f"export LOCUST_THROUGHPUT_PATH={shlex.quote(THROUGHPUT_ENDPOINT_PATH)}",
        f"export LOCUST_VERIFY_TLS={'true' if LOCUST_VERIFY_TLS_BOOL else 'false'}",
        f"export LOCUST_CONNECT_TIMEOUT_S={CONNECT_S}",
        f"export LOCUST_READ_TIMEOUT_S={READ_S}",
        f"export LOCUST_WAIT_TIME_S={LOCUST_WAIT_TIME_SEC}",
        'export PATH="$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH"',
    ]
    if TARGETS_CSV:
        lines += [f"export LOCUST_TARGETS={shlex.quote(TARGETS_CSV)}", "export LOCUST_NAME_BY_VIP=true"]
    else:
        lines += ["unset LOCUST_TARGETS", "export LOCUST_NAME_BY_VIP=false"]
    return "\n".join(lines)

# Master: run in tmux, foreground (exec), no '&'
run_ui_sh = f"""#!/usr/bin/env bash
set -euo pipefail
{_env_exports()}
cd {shlex.quote(LOCUST_WORKDIR)}
exec python3 -m locust -f locustfile.py --master --master-bind-host {MASTER_BIND_HOST} \
  --web-host 0.0.0.0 --web-port {UI_WEB_PORT} \
  > {shlex.quote(LOCUST_WORKDIR)}/ui_master.log 2>&1
"""
sftp_write(MASTER, run_ui_sh, f"{LOCUST_WORKDIR}/run_ui.sh", mode=0o755)
# Ensure clean session then start
ssh_exec(MASTER, "tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true", timeout=10)
_, err, rc = ssh_exec(MASTER, f"tmux new -d -s ui_master {shlex.quote(LOCUST_WORKDIR)}/run_ui.sh", timeout=20)
print(f"[master@{MASTER}] ui_master started (tmux rc={rc})")
if err: print("[stderr]", err)

# Compute workers per host
def _detect_nproc(host: str) -> int:
    out, _, _ = ssh_exec(host, "nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1", timeout=10)
    try:
        return max(1, int((out or "1").strip()))
    except Exception:
        return 1

def _workers_for_host(host: str) -> int:
    if WORKERS_PER_HOST_ENV != "auto":
        try:
            return max(1, int(WORKERS_PER_HOST_ENV))
        except Exception:
            return 1
    n = _detect_nproc(host)
    w = max(1, n - max(0, CPU_RESERVE))
    return max(MIN_WORKERS_PER_HOST, min(MAX_WORKERS_PER_HOST, w))

per_host_workers = {h: _workers_for_host(h) for h in GEN_IPS}
print(f"Planned workers per host: {per_host_workers} | total={sum(per_host_workers.values())}")

# Per-host start script using nohup; one SSH per host; parallel across hosts
def spawn_on_host(host: str) -> Tuple[str, int, str]:
    n = per_host_workers[host]
    start_workers_sh = f"""#!/usr/bin/env bash
set -euo pipefail
{_env_exports()}
cd {shlex.quote(LOCUST_WORKDIR)}
echo "Spawning {n} worker(s) to master {MASTER_ADDR_FOR_WORKERS_RAW} at $(date -u)"
for i in $(seq 1 {n}); do
  nohup python3 -m locust -f locustfile.py --worker --master-host {MASTER_ADDR_FOR_WORKERS} > locust-worker-$i.log 2>&1 &
  sleep 0.05
done
echo WORKERS_STARTED $(date -u)
"""
    remote_path = f"{LOCUST_WORKDIR}/start_workers.sh"
    sftp_write(host, start_workers_sh, remote_path, mode=0o755)
    out, err, rc = ssh_exec(host, f"nohup {shlex.quote(remote_path)} > {shlex.quote(LOCUST_WORKDIR)}/start_workers.out 2>&1 & echo OK", timeout=15)
    return host, (0 if "OK" in out else (rc if rc is not None else 1)), err

print(f"Spawning workers (parallel hosts; concurrency={min(HOST_SPAWN_CONCURRENCY, len(GEN_IPS))}) ...")
with concurrent.futures.ThreadPoolExecutor(max_workers=min(HOST_SPAWN_CONCURRENCY, len(GEN_IPS))) as ex:
    for host, rc, err in ex.map(spawn_on_host, GEN_IPS):
        if rc != 0 or err:
            print(f"[host-spawn][{host}] rc={rc} err={err}")

# Readiness: wait for master UI listener
def wait_master_ui(timeout_s=90, interval_s=1.5):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        out, _, _ = ssh_exec(MASTER, f"ss -lntp | grep -E ':{UI_WEB_PORT}\\s' || true", timeout=10)
        if out.strip():
            return True, out.strip()
        time.sleep(interval_s)
    return False, ""

ok, listen_info = wait_master_ui()
print("\nMaster UI readiness:", "READY" if ok else "NOT READY")
if listen_info:
    print(listen_info)
else:
    log_tail, _, _ = ssh_exec(MASTER, f'tail -n 60 "{LOCUST_WORKDIR}/ui_master.log" 2>/dev/null || true', timeout=10)
    if log_tail:
        print("Master log (tail):")
        print(log_tail)

# Readiness: wait for workers per host
def wait_workers(host: str, expected: int, timeout_s=180, interval_s=2):
    deadline = time.time() + timeout_s
    last = "0"
    while time.time() < deadline:
        cnt, _, _ = ssh_exec(host, r"pgrep -f 'locust.*--worker' | wc -l || true", timeout=10)
        last = (cnt or "0").strip()
        try:
            if int(last) >= expected:
                return True, last
        except Exception:
            pass
        time.sleep(interval_s)
    return False, last

print("\nWorker readiness per host:")
for hip in GEN_IPS:
    exp = per_host_workers[hip]
    ready, seen = wait_workers(hip, exp, timeout_s=180, interval_s=2)
    print(f"  {hip}: {'READY' if ready else 'NOT READY'} (seen={seen}, expected~{exp})")
    if not ready:
        wlog, _, _ = ssh_exec(hip, 'ls -1t "$HOME/locustwork"/locust-worker-*.log 2>/dev/null | head -n1 | xargs -r tail -n 30 || true', timeout=10)
        if wlog:
            print("    worker log (tail):")
            print("    " + "\n    ".join(wlog.splitlines()))

# UI URL and SSH tunnel
print("\nLocust UI:")
print(f"  Direct URL (public):   http://{MASTER}:{UI_WEB_PORT}")
print("  Recommended (tunnel) from your laptop:")
print(f'    ssh -i "{SSH_KEY}" -o ExitOnForwardFailure=yes -N -L {UI_WEB_PORT}:127.0.0.1:{UI_WEB_PORT} opc@{MASTER}')
print("  Then open:")
print(f"    http://127.0.0.1:{UI_WEB_PORT}")

print("\nCell 11 complete: Master up; workers spawned with one SSH per host; readiness checked.")
print("NEXT: Use the UI to start a test, or proceed to Cell 12 to stop the UI/workers, and Cell 13 to teardown.")


In [ ]:
# Cell 12 — Objective: Stop Locust UI and workers (tmux ui_master + kill workers on all hosts) with parallelism
# - Kills the ui_master tmux session on the master.
# - Kills all worker processes across generators in parallel.
# - Prints remaining worker counts per host for confirmation.
# - SSH control-plane is IPv4 by default; honors USE_SSH_IPV6=true if set and IPv6 list is available.

import os, json, paramiko, concurrent.futures
from pathlib import Path
from typing import List, Tuple

def _load_list_env(name: str, default="[]") -> List[str]:
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

GEN_IPS_V4 = _load_list_env("GEN_IPS_V4_JSON")
GEN_IPS_V6 = _load_list_env("GEN_IPS_V6_JSON")
USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")
GEN_IPS = GEN_IPS_V6 if (USE_SSH_IPV6 and GEN_IPS_V6) else GEN_IPS_V4
if not GEN_IPS:
    raise RuntimeError("No generator hosts available. Run previous cells to provision and capture outputs.")

MASTER = GEN_IPS[0]
SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip() or "~/.ssh/id_rsa")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

if not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not found: {SSH_KEY}")

def _load_pkey(path: str, pw: str | None):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key: {path}")

def ssh_exec(host: str, command: str, timeout: int = 60) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8", "ignore") or "").strip()
    err = (stderr.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

def kill_ui_master():
    print("Stopping ui_master tmux on master ...")
    ssh_exec(MASTER, "tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true", timeout=15)
    print("ui_master stopped.")

def kill_workers_on_host(host: str) -> Tuple[str, int]:
    # Kill both python3 -m locust and bare 'locust' worker invocations (best effort)
    cmd = r"""bash -lc '
pkill -f "python3 -m locust .*--worker" 2>/dev/null || true
pkill -f "locust .*--worker" 2>/dev/null || true
pgrep -f "locust.*--worker" | wc -l || true
'"""
    out, _, _ = ssh_exec(host, cmd, timeout=20)
    try:
        remaining = int((out or "0").splitlines()[-1].strip())
    except Exception:
        remaining = 0
    return host, remaining

def stop_all_workers_parallel(concurrency: int = 8):
    print("Stopping workers on all hosts ...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(concurrency, len(GEN_IPS))) as ex:
        for host, remaining in ex.map(kill_workers_on_host, GEN_IPS):
            print(f"  {host}: remaining workers≈{remaining}")

# One-shot convenience
def stop_everything():
    kill_ui_master()
    stop_all_workers_parallel(concurrency=8)

# Execute stops now
stop_everything()

print("\nCell 12 complete: UI and workers stopped. You can re-run Cell 11 to start again, or proceed to Cell 13 to teardown.")


In [ ]:
# Cell 13 — Objective: Teardown (guarded)
# - Set TEARDOWN_CONFIRM=True to destroy resources created by this run.
# - Uses terraform.tfvars from Cell 6 to ensure correct context.
# - Does not print or handle any secrets.

TEARDOWN_CONFIRM = True  # Set False to skip

# Optional: quick check for terraform binary (informative)
try:
    import shutil
    if shutil.which("terraform") is None:
        print("Warning: 'terraform' not found in PATH. Install Terraform and re-run this cell.")
except Exception:
    pass

if TEARDOWN_CONFIRM:
    # Optional higher parallelism for destroy
    !terraform destroy -parallelism=20 -auto-approve -var-file=terraform.tfvars
    print("Cell 13 complete: All Terraform resources destroyed.")
else:
    print("Teardown guard is False. Set TEARDOWN_CONFIRM=True to destroy resources.")
    print("Cell 13 complete: No teardown executed.")